# Writing our own einsum

- Problem: how to handle more than 1 undefined indices?
- Solution idea: use itertools.product

In [ ]:
import numpy as np


def einsum(expression: str, *tensors: np.ndarray) -> np.ndarray:
    [input_indices, output_index] = expression.split("->", 1)
    input_indices = [x.strip() for x in input_indices.split(",")]
    output_index = output_index.strip()
    assert len(input_indices) == len(tensors)
    
    lengths: dict[str, int] = {}
    for tensor_i, input_index in enumerate(input_indices):
        for char_i, char in enumerate(input_index):
            if char in lengths:
                assert lengths[char] == tensors[tensor_i].shape[char_i]
            else:
                lengths[char] = tensors[tensor_i].shape[char_i]

    out = np.empty(tuple([lengths[x] for x in output_index]))
    
    itr = np.nditer(out, flags=["multi_index"])
    for _ in itr:
        defined_index = {}
        for zi, z in enumerate(output_index):
            defined_index[z] = itr.multi_index[zi]

        undefined_indices: set[str] = set()
        for tensor_i, input_index in enumerate(input_indices):
            for zi, z in enumerate(input_index):
                if z not in defined_index:
                    undefined_indices.add(z)

        undefined_index = "<None>"
        lengths[undefined_index] = 1
        if len(undefined_indices) == 1:
            undefined_index = undefined_indices.pop()
        elif len(undefined_indices) > 1:
            raise TypeError("uhhh i don't know how to handle this")

        acc = 0
        for undef in range(lengths[undefined_index]):
            macc = 1
            for tensor_i, tensor in enumerate(tensors):
                idx = tuple([
                    defined_index.get(x, undef)
                    for x in input_indices[tensor_i]
                ])
                macc *= tensor[idx]
            acc += macc
        out[itr.multi_index] = acc
    
    return out



a = 6 - np.arange(6).reshape(3,2)
b = np.arange(6).reshape(2,3)


einsum("mn, np -> mp", a, b), a @ b

(array([[15., 26., 37.],
        [ 9., 16., 23.],
        [ 3.,  6.,  9.]]),
 array([[15, 26, 37],
        [ 9, 16, 23],
        [ 3,  6,  9]]))

In [65]:
einsum("ij->ji",a), a.T

(array([[6., 4., 2.],
        [5., 3., 1.]]),
 array([[6, 4, 2],
        [5, 3, 1]]))